# Nettoyage & Préparation — Base Membres

Ce notebook applique le pipeline de nettoyage, de typage et de standardisation métier sur la base consolidée issue du feature engineering (`base_membres.xlsx`).

### 1. Entrées & Sorties

| Type | Fichier / Répertoire | Description |
| :--- | :--- | :--- |
| **Entrée** | `../data/base_membres.xlsx` | Table brute issue de l'étape 01 (1 ligne / contact avec métriques agrégées). |
| **Sortie (Données)** | `../data/clean_base_membres.xlsx` | Table nettoyée, typée et prête pour l'agrégation adhérent ou la modélisation. |
| **Sortie (Qualité)** | `../outputs/rapports/clean_rapport_base_membres_*.html` | Rapports interactifs de distribution et d'intégrité (`skrub.TableReport`). |

### 2. Principales Transformations Métier

* **Consolidation hiérarchique** : propagation des rôles détaillés (`Is CEO`, `Is director`, etc.) vers les niveaux d'influence `C-Level`, `M-Level` et `F-Level`, puis génération des indicateurs booléens nullables (`bool_Is *LEVEL`).
* **Taxonomie des départements** : harmonisation des 17 intitulés bruts vers une nomenclature unifiée (Marketing, Médias, Juridique, Insights, Impact, Direction générale, etc.) avec gestion des statuts incertains.
* **Standardisation des types** :
  * Dates ISO et dates CRM au format `datetime64`.
  * Consentements et présences en `boolean` nullable.
  * Métriques de volume et de récence converties en entiers (`Int64`).
* **Nettoyage du périmètre** :
  * Élimination des identifiants techniques dupliqués et colonnes redondantes.
  * Exclusion des structures internes / tests (ex. ID Union des Marques `1537`).
  * Arbitrage des statuts de contact (`Prospect` rattaché à une organisation membre).

### 3. Pipeline d'Exécution
1. Consolidation hiérarchique (C-Level / M-Level)
2. Typage strict (Dates, Booléens, Entiers)
3. Regroupement thématique des départements
4. Encodage catégoriel & Dummies sectorielles
5. Purge des colonnes obsolètes & Filtrage des organisations hors périmètre
6. Résolution des statuts (Current, Former, Prospects)
7. Exports & Audit qualité : génération du fichier Excel et des rapports de distribution `skrub.TableReport`.

# Imports & configuration

In [ ]:
import sys
import pandas as pd
from pathlib import Path
from skrub import TableReport

# Accès aux utilitaires du projet
sys.path.append(str(Path.cwd().parent))
from src.utils import convert_to_date_day_first, convert_to_date_year_first, convert_to_bool, convert_to_cat, convert_to_int

# ── Paramètres centralisés ──────────────────────────────────────────────────
DATA_PATH = "../data/base_membres.xlsx"

# Colonnes de dates à convertir
DATE_COLS = [
    "User last online",
    "Organisation - First contact date",
    "Function Start date",
]

# Colonnes détaillées "Is *" → colonnes consolidées "Is *LEVEL"
IS_LEVEL_MAPPING = {
    "Is C LEVEL": ["Is CEO / President", "Is vice president", "Is director",
                   "Is managing director", "Is deputy managing director"],
    "Is M LEVEL": ["Is manager"],
    # "Is F LEVEL" n'a pas de source connue : laissée telle quelle
}
LEVEL_COLS = ["Is F LEVEL", "Is M LEVEL", "Is C LEVEL"]

# Colonnes de département (dummies)
DEPARTMENT_COLS = [
    "Direction générale", "Juridique / Fiscal", "RH", "Stratégie / Etudes",
    "Communication", "Publicité", "RSE", "Affaires Publiques", "Marketing",
    "Production publicitaire / Création", "Finance", "Marketing Client",
    "Achats", "Digital", "Commercial", "Marketing Produit", "Marketing opérationnel",
]
Nouveau_dep = {
    "Direction générale" : "Direction générale",
    "Juridique / Fiscal" : "Juridique",
    "RH" : "RH",
    "Stratégie / Etudes" : "Insights",
    "Communication" : "Marketing",
    "Publicité" : "Médias",
    "RSE" : "Impact",
    "Affaires Publiques" : "Affaires Publiques",
    "Marketing" : "Marketing",
    "Production publicitaire / Création" : "Médias",
    "Finance" : "Juridique",
    "Marketing Client" : "Marketing",
    "Achats" : "Médias",
    "Digital" : "Performance digitale",
    "Commercial" : "Marketing",
    "Marketing Produit" : "Marketing",
    "Marketing opérationnel" : "Marketing"
}

# Colonnes booléennes (communications)
BOOL_COMM_COLS = [
    "Communication - Communautés",
    "Communication - Newsletters",
    "Communication - Partner communications",
    "Communication - Veille juridique",
    "Meeting&Events speaker",
    "Organisation - Communication - Communautés",
]

# Colonnes catégorielles
CAT_COLS = [
    "Membership status (description)",
    "Organisation - Company position",
    "Organisation - Membership status",
    "Organisation - Sector of Activities",
    "Recipient Status",
    "Company subscription",
]

# Colonnes à supprimer en fin de pipeline
COLS_TO_DROP = [
    "Company position (description)",
    "Notification on new comment",
    "Notifications on new post",
    "Nr. open invoices",
    "Amount open invoices",
    "Relation ID",
    "Relation ID_mailing",
    "unique_mailing_tags_count_by_relation",
    "Attendee Relation ID",
    "unique_meeting_tag_count_by_relation",
    "ID CRM",
    # Colonnes Is * détaillées remplacées par les consolidées
    "Is CEO / President", "Is deputy managing director", "Is director",
    "Is F LEVEL", "Is M LEVEL", "Is manager", "Is managing director",
    "Is subcompany", "Is vice president", "Is C LEVEL",
]

# Chargement des données

In [ ]:
df = pd.read_excel(DATA_PATH).copy()
print(f"Dimensions initiales : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.head(3)

# 1. Consolidation hiérarchique (C-Level / M-Level)

Logique :
1. Propager les colonnes détaillées (`Is CEO`, `Is director`…) vers les colonnes consolidées (`Is C LEVEL`, `Is M LEVEL`)
2. Pour les contacts ayant au moins un niveau renseigné, remplir les autres niveaux par `"Non Concerné"`
3. Créer les colonnes `bool_Is *LEVEL` (True / False / NA)

In [ ]:
def consolider_niveaux(df, mapping, level_cols):
    """
    Propage les colonnes Is * détaillées vers les colonnes Is *LEVEL consolidées.
    
    Parameters
    ----------
    df : pd.DataFrame
    mapping : dict  {col_cible: [cols_sources]}
    level_cols : list  colonnes consolidées à analyser ensemble
    """
    nan_avant = df[level_cols].isna().all(axis=1).sum()

    # Propagation source → cible
    for col_cible, sources in mapping.items():
        for source in sources:
            if source not in df.columns:
                continue
            masque = df[col_cible].isna() & df[source].notna()
            df.loc[masque, col_cible] = df.loc[masque, source]

    # Contacts avec au moins un niveau → "Non Concerné" pour les autres
    a_un_niveau = df[level_cols].notna().any(axis=1)
    df.loc[a_un_niveau, level_cols] = (
        df.loc[a_un_niveau, level_cols].fillna("Non Concerné")
    )

    nan_apres = df[level_cols].isna().all(axis=1).sum()
    print(f"Contacts sans aucun niveau — avant : {nan_avant}  →  après : {nan_apres}  (−{nan_avant - nan_apres})")
    return df


def creer_bool_niveaux(df, level_cols):
    """
    Crée des colonnes booléennes préfixées 'bool_' à partir des colonnes Is *LEVEL.
    True  = niveau renseigné, False = "Non Concerné", NA = inconnu.
    """
    level_df = df[level_cols].copy()
    bool_df = level_df.apply(
        lambda col: col.map(
            lambda x: pd.NA if pd.isna(x) else (False if x == "Non Concerné" else True)
        )
    ).astype("boolean")
    bool_df.columns = [f"bool_{c}" for c in level_cols]
    return pd.concat([df, bool_df], axis=1)


# ── Exécution ────────────────────────────────────────────────────────────────
df = consolider_niveaux(df, IS_LEVEL_MAPPING, LEVEL_COLS)
df = creer_bool_niveaux(df, LEVEL_COLS)

bool_cols = [c for c in df.columns if c.startswith("bool_Is")]
print("\nDistribution des colonnes bool_Is *LEVEL :")
df[bool_cols].apply(lambda c: c.value_counts(dropna=False)).T

# 2. Typage strict (Dates, Booléens, Entiers)

## 1. Conversion des colonnes de dates

In [ ]:
for col in DATE_COLS:
    if col in df.columns:
        convert_to_date_day_first(df, col)
convert_to_date_year_first(df, "Creation date")

print("Types après conversion :")
print(df[DATE_COLS + ["Creation date"]].dtypes)
print("\nNaN par colonne :")
print(df[DATE_COLS + ["Creation date"]].isna().sum())

## 2. Conversion des colonnes booléennes (communications)

In [ ]:
for col in BOOL_COMM_COLS:
    if col in df.columns:
        convert_to_bool(df, col)

print("Types après conversion :")
print(df[BOOL_COMM_COLS].dtypes)

# 3. Regroupement thématique des départements

- Les colonnes département contiennent `1.0` (présent) ou `NaN`
- Si une ligne a au moins un département renseigné, les autres sont mis à `False`
- Sinon, toutes les colonnes restent `NA`

In [ ]:
def encoder_departements(df, dept_cols):
    """
    Crée des colonnes 'department_<nom>' booléennes regroupées selon Nouveau_dep.
    
    Règles de regroupement pour chaque groupe :
    - si au moins un True -> True
    - si tous False -> False
    - si tous NaN -> NaN
    - sinon (mélange False + NaN sans True) -> NaN (incertain)
    """
    # colonnes d'origine existantes
    dept_df = df[dept_cols].copy()

    # normaliser valeurs 1.0/0.0/NaN en True/False/NaN
    dept_bool = dept_df.map(lambda x: True if x == 1.0 else (False if x == 0.0 else pd.NA))
    dept_bool.columns = [f"department_{c}" for c in dept_bool.columns]
    # si un département est True, les NaN deviennent False
    has_true = dept_bool.eq(True).any(axis=1)

    dept_bool = dept_bool.mask(
        has_true.values[:, None] & dept_bool.isna(),
        False
    )
    df = pd.concat([df, dept_bool], axis=1)

    # construire colonnes regroupées selon Nouveau_dep
    grouped_cols = {}
    for orig, target in Nouveau_dep.items():
        if orig not in dept_cols:
            print(orig)
            continue
        col_name = f"department_{target.replace(' ', '_').replace('/', '_')}"
        grouped_cols.setdefault(col_name, []).append(f"department_{orig}")

    created = []
    for grp_col, src_cols in grouped_cols.items():
        # ne garder que les sources présentes dans le df
        src_cols = [c for c in src_cols if c in df.columns]
        if not src_cols:
            continue
        grp_df = df[src_cols]

        any_true = grp_df.eq(True).any(axis=1)
        all_false = grp_df.eq(False).all(axis=1)

        res = pd.Series(pd.NA, index=df.index, dtype="boolean")
        res[any_true] = True
        res[all_false] = False
        df[grp_col] = res
        convert_to_bool(df, grp_col)
        created.append(grp_col)

    # suppression des colonnes department_{orig} laissées en double
    orig_dept_cols = [
        f"department_{c}" 
        for c in dept_cols 
        if f"department_{c}" in df.columns 
        and f"department_{c}" not in created
    ]
    df = df.drop(columns=orig_dept_cols)

    nb_sans_dept = df[created].isna().all(axis=1).sum()
    print(f"Contacts sans aucun département renseigné (après regroupement) : {nb_sans_dept}")
    return df


# ── Exécution ────────────────────────────────────────────────────────────────
dept_cols_existants = [c for c in DEPARTMENT_COLS if c in df.columns]
df = encoder_departements(df, dept_cols_existants)

Direction générale => on garde  
Juridique / Fiscal => Juridique  
RH => on garde  
Stratégie / Études => Insights  
Communication => Marketing  
Publicité => Médias  
RSE => Impact  
Affaires Publiques => Affaires Publiques  
Marketing => Marketing  
Production publicitaire / Création => Médias  
Finance => Juridique  
Marketing Client => Marketing  
Achats => Médias  
Digital => Performance digitale  
Commercial => Marketing  
Marketing Produit => Marketing  
Marketing opérationnel => Marketing  

# 4. Encodage catégoriel & Dummies sectorielles

In [ ]:
for col in CAT_COLS:
    if col in df.columns:
        convert_to_cat(df, col)

print("Modalités par colonne catégorielle :")
for col in CAT_COLS:
    if col in df.columns:
        modalites = df[col].cat.categories.tolist()
        print(f"  {col} ({len(modalites)}) : {modalites}")

**Traitement de la colonne `VIP`**

La colonne `VIP` est une présence/absence : `notna()` → `True`, sinon `False`.

In [ ]:
if "VIP" in df.columns:
    df["VIP"] = df["VIP"].notna().astype("boolean")
    print("Distribution VIP :", df["VIP"].value_counts(dropna=False).to_dict())

**variable secteur d'activité**

In [ ]:
if "Organisation - Sector of Activities" in df.columns:
    secteur_dummies = pd.get_dummies(
        df["Organisation - Sector of Activities"],
        prefix="secteur",
        prefix_sep="_",
        dummy_na=False
    )
    df = pd.concat([df, secteur_dummies], axis=1)
    print(f"Colonnes dummies créées : {secteur_dummies.columns.tolist()}")
else:
    print("Colonne 'Organisation - Sector of Activities' introuvable.")

# 5. Purge des colonnes obsolètes & Filtrage des organisations hors périmètre

In [ ]:
cols_presentes = [c for c in COLS_TO_DROP if c in df.columns]
cols_absentes  = [c for c in COLS_TO_DROP if c not in df.columns]

df = df.drop(columns=cols_presentes)

print(f"Colonnes supprimées ({len(cols_presentes)}) : {cols_presentes}")
if cols_absentes:
    print(f"Non trouvées (ignorées) : {cols_absentes}")
print(f"\nDimensions finales : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

**Suppression des contacts à ne pas garder**

In [ ]:
ids_to_remove = {145774, 183276, 168765, 1537}
col = "Organisation - Relation ID"

if col in df.columns:
    before = len(df)
    df = df[~df[col].isin(ids_to_remove)].copy()
    print(f"Lignes supprimées : {before - len(df)}")
else:
    print(f"Colonne '{col}' introuvable.")

**Nettoyage de la base de données meetings**

In [ ]:
a_convertir = [
    "registered_count_by_relation",
    "present_count_by_relation",
    "days_since_last_participation",
    "days_since_last_registration",
    "cancelled_count_by_relation",
    "opted_out_count_by_relation",
    "reserve_list_count_by_relation",
    "no_reaction_count_by_relation"
]
for col in a_convertir :
    convert_to_int(df, col)

**Nettoyage de la base de données click**

In [ ]:
convert_to_int(df, "clicks_count_by_relation")

# 6. Résolution des statuts (Current, Former, Prospects)

**Suppression de certaines lignes**
1. Toutes les lignes pour lesquelles Membership status (description) est différents de Former Member, Current Member et Prospect

In [ ]:
allowed_status = {"Former Member", "Current Member", "Prospect"}

if "Membership status (description)" in df.columns:
    before = len(df)
    df = df[df["Membership status (description)"].isin(allowed_status)].copy()
    removed = before - len(df)
    print(f"Lignes supprimées : {removed}")
else:
    print("La colonne 'Membership status (description)' est introuvable.")

## Gestion des prospects (Arbitrage interactif / Optionnel)

### À quoi sert cette étape ?
Certains contacts sont étiquetés avec le statut individuel `Prospect` alors que leur organisation est déjà `Current Member` ou `Former Member`.
Cette cellule permet d'arbitrer ces incohérences :
1. **Reconnaissance automatique** : si le nom de l'organisation est inclus dans le nom de l'entreprise du contact, le statut du contact prend automatiquement celui de son organisation.
2. **Reconnaissance semi-automatique interactive** : pour les cas restants, elle affiche les correspondances potentielles et propose interactivement (`y/N`) de valider ou non le rattachement.



### Comment la lancer ?
> **Note importante :** Pour éviter de bloquer le notebook lors d'un **"Run All"** (l'instruction `input()` attendant une saisie clavier), cette cellule est **désactivée par défaut**.

Pour l'exécuter manuellement :
1. Dans la cellule de code ci-dessous, modifiez la variable :  
   `RUN_INTERACTIVE_PROSPECTS = True`
2. Sélectionnez la cellule puis appuyez sur **Maj + Entrée** (`Shift + Enter`).
3. Répondez `o` (ou `y`) dans le champ de saisie pour valider chaque correspondance proposée.

In [ ]:
# ==============================================================================
# PARAMÈTRE D'EXÉCUTION
# Passer à True pour débloquer l'arbitrage interactif des prospects.
# Laisser à False lors d'un "Run All" pour ne pas bloquer le notebook.
# ==============================================================================
RUN_INTERACTIVE_PROSPECTS = False


if not RUN_INTERACTIVE_PROSPECTS:
    print("ℹ️ Cellule ignorée lors du 'Run All' pour éviter de bloquer sur input().")
    print("👉 Pour la lancer : passez 'RUN_INTERACTIVE_PROSPECTS = True' et exécutez la cellule manuellement (Shift + Enter).")

else:
    import difflib
    import re

    def normalize_name(name):
        if pd.isna(name):
            return ""
        text = str(name).lower()
        text = re.sub(r"[^\w\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def is_relation_in_company(relation, company):
        rel = normalize_name(relation)
        comp = normalize_name(company)
        return bool(rel and comp and rel in comp)

    def similarity_score(a, b):
        return difflib.SequenceMatcher(
            None, normalize_name(a), normalize_name(b)
        ).ratio()

    # Sélection des prospects
    prospect_mask = df["Membership status (description)"] == "Prospect"
    prospects = df.loc[prospect_mask, [
        "Organisation - Relation name",
        "Company name",
        "Organisation - Membership status",
        "Membership status (description)"
    ]].copy()

    if prospects.empty:
        print("Aucun contact avec le statut 'Prospect' trouvé.")
    else:
        # 1. Reconnaissance automatique
        auto_recognized = prospects.apply(
            lambda row: is_relation_in_company(
                row["Organisation - Relation name"], row["Company name"]
            ),
            axis=1
        )
        auto_idx = prospects[auto_recognized].index

        print(f"✓ Reconnu automatiquement : {len(auto_idx)} lignes")
        if len(auto_idx) > 0:
            print(df.loc[auto_idx, [
                "Organisation - Relation name",
                "Company name",
                "Organisation - Membership status"
            ]].to_string(index=False))
            
            df.loc[auto_idx, "Membership status (description)"] = (
                df.loc[auto_idx, "Organisation - Membership status"].astype(str)
            )

        # 2. Arbitrage manuel interactif
        remaining = prospects.loc[~auto_recognized].copy()
        if remaining.empty:
            print("\nAucun prospect restant à arbitrer après reconnaissance automatique.")
        else:
            print(f"\n--- Début de l'arbitrage manuel ({len(remaining)} contacts restants) ---")
            for idx, row in remaining.iterrows():
                print("\n------------------------------------------------------------")
                print(f"Index : {idx}")
                print("Organisation - Relation name :", row["Organisation - Relation name"])
                print("Company name                 :", row["Company name"])
                print("Organisation - Membership status :", row["Organisation - Membership status"])
                
                answer = input("Valider comme similaire ? [y/N] : ").strip().lower()
                if answer in {"y", "yes", "o", "oui"}:
                    df.loc[idx, "Membership status (description)"] = row["Organisation - Membership status"]
                    print("  → Modifié selon le statut de l'organisation")
                else:
                    print("  → Conservé en 'Prospect'")

            print("\n✓ Arbitrage interactif terminé.")

# 7. Exports & Audit qualité

In [ ]:
OUTPUT_EXCEL = "../data/base_membres_propre.xlsx"
OUTPUT_HTML  = "../outputs/rapports/rapport_base_membres_propre.html"

chunk_size = 30
reports = []

for part, start in enumerate(range(0, len(df.columns), chunk_size), start=1):
    cols = df.columns[start : start + chunk_size]
    chunk_report = TableReport(df[cols])
    output_part_html = OUTPUT_HTML.replace(".html", f"_part{part:02d}.html")
    chunk_report.write_html(output_part_html)
    print(f"Rapport HTML écrit : {output_part_html}")
    reports.append(chunk_report)

report = reports[0]

# Export Excel
df.to_excel(OUTPUT_EXCEL, index=False)
print(f"Export Excel écrit : {OUTPUT_EXCEL}")

report